<a href="https://colab.research.google.com/github/Victorhcgo/202507_ProyectoModeloseleccionCCC/blob/main/riesgo_credito_german_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 Scoring de Riesgo de Crédito — German Credit Dataset
### Preparación Prueba Técnica: Analista de Riesgo de Crédito y Seguimiento

**Flujo del notebook:**
1. Instalación y carga del dataset
2. Exploración y limpieza de datos
3. Análisis exploratorio (EDA)
4. Regresión logística + diagnóstico de supuestos
5. Evaluación del modelo (AUC-ROC, KS, matriz de confusión)
6. Regresión OLS + pruebas econométricas
7. Series de tiempo: índice de mora simulado
8. Cálculo de Pérdida Esperada (EL = PD × LGD × EAD)


## 0. Instalación de librerías

In [ ]:
# Ejecutar solo una vez en Colab
!pip install ucimlrepo statsmodels scikit-learn pandas numpy matplotlib seaborn -q

## 1. Carga del dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')

# Cargar German Credit Dataset desde UCI
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=144)

X_raw = dataset.data.features
y_raw = dataset.data.targets

# Unir en un solo DataFrame
df = pd.concat([X_raw, y_raw], axis=1)
df.columns = list(X_raw.columns) + ['class']

# La variable objetivo: 1 = buen pagador, 2 = mal pagador → convertir a 0/1
df['default'] = (df['class'] == 2).astype(int)
df.drop('class', axis=1, inplace=True)

print(f'Shape: {df.shape}')
print(f'Tasa de default: {df["default"].mean():.1%}')
df.head()

## 2. Exploración y limpieza

In [ ]:
# Tipos de variables y nulos
print('=== INFO ===')
print(df.info())
print('\n=== NULOS ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Estadísticas descriptivas de variables numéricas
numericas = df.select_dtypes(include=[np.number]).columns.tolist()
print('Variables numéricas:', numericas)
df[numericas].describe().round(2)

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Conteo
counts = df['default'].value_counts()
axes[0].bar(['Buen pagador (0)', 'Mal pagador (1)'], counts.values, color=['#1a6bab', '#d64045'])
axes[0].set_title('Distribución de default', fontweight='bold')
axes[0].set_ylabel('Cantidad de clientes')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Duración del crédito por default
df.groupby('default')['duration'].plot(kind='hist', alpha=0.6, bins=20, ax=axes[1],
                                        label=['Buen pagador', 'Mal pagador'],
                                        color=['#1a6bab', '#d64045'])
axes[1].set_title('Duración del crédito por tipo de cliente', fontweight='bold')
axes[1].set_xlabel('Duración (meses)')
axes[1].legend(['Buen pagador', 'Mal pagador'])

plt.tight_layout()
plt.show()
print('\n💡 Observación: Los créditos de mayor duración tienden a tener mayor tasa de default.')

In [ ]:
# Preparar variables para el modelo
# Variables numéricas disponibles en este dataset
vars_num = ['duration', 'credit_amount', 'installment_commitment',
            'residence_since', 'age', 'existing_credits', 'num_dependents']

# Mantener solo las numéricas + target para los modelos econométricos
df_model = df[vars_num + ['default']].copy()

print('Variables del modelo:')
print(df_model.describe().round(2))

## 3. Análisis exploratorio (EDA)

In [ ]:
# Tasa de default por segmentos clave
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Por rango de monto
df_model['rango_monto'] = pd.cut(df_model['credit_amount'],
                                   bins=[0, 2000, 5000, 10000, 20000],
                                   labels=['Bajo', 'Medio', 'Alto', 'Muy alto'])
tasa_monto = df_model.groupby('rango_monto')['default'].mean()
axes[0].bar(tasa_monto.index, tasa_monto.values * 100, color='#1a6bab')
axes[0].set_title('Tasa default por monto', fontweight='bold')
axes[0].set_ylabel('% default')
axes[0].set_ylim(0, 50)

# Por rango de duración
df_model['rango_duracion'] = pd.cut(df_model['duration'],
                                      bins=[0, 12, 24, 48, 72],
                                      labels=['≤12m', '13-24m', '25-48m', '>48m'])
tasa_dur = df_model.groupby('rango_duracion')['default'].mean()
axes[1].bar(tasa_dur.index, tasa_dur.values * 100, color='#d64045')
axes[1].set_title('Tasa default por duración', fontweight='bold')
axes[1].set_ylabel('% default')
axes[1].set_ylim(0, 50)

# Correlación
corr = df_model[vars_num + ['default']].corr()
sns.heatmap(corr[['default']].drop('default').sort_values('default'),
            annot=True, fmt='.2f', cmap='RdYlBu_r', ax=axes[2],
            vmin=-0.5, vmax=0.5)
axes[2].set_title('Correlación con default', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Regresión Logística — Modelo de Scoring

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, roc_curve, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

X = df_model[vars_num]
y = df_model['default']

# Dividir en entrenamiento y prueba (estratificado para mantener proporción de defaults)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Escalar (importante para regresión logística)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Entrenar modelo
logit = LogisticRegression(random_state=42, max_iter=1000)
logit.fit(X_train_sc, y_train)

# Probabilidades predichas
prob_train = logit.predict_proba(X_train_sc)[:, 1]
prob_test  = logit.predict_proba(X_test_sc)[:, 1]

print('=== DESEMPEÑO DEL MODELO ===')
print(f'AUC-ROC Train : {roc_auc_score(y_train, prob_train):.4f}')
print(f'AUC-ROC Test  : {roc_auc_score(y_test, prob_test):.4f}')
print()
print(classification_report(y_test, logit.predict(X_test_sc),
                             target_names=['Buen pagador', 'Mal pagador']))

In [ ]:
# Curva ROC + Matriz de confusión
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
fpr, tpr, _ = roc_curve(y_test, prob_test)
auc = roc_auc_score(y_test, prob_test)
axes[0].plot(fpr, tpr, color='#1a6bab', lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Modelo aleatorio')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#1a6bab')
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].set_title('Curva ROC — Modelo de Default', fontweight='bold')
axes[0].legend()

# Matriz de confusión
cm = confusion_matrix(y_test, logit.predict(X_test_sc))
disp = ConfusionMatrixDisplay(cm, display_labels=['Buen pagador', 'Mal pagador'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Matriz de Confusión', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n💡 Interpretación:')
print(f'  - AUC = {auc:.3f}: El modelo discrimina correctamente en el {auc*100:.1f}% de los casos.')
print('  - Falsos Negativos (predice buen pagador, era mal pagador) = mayor riesgo para el banco.')

In [ ]:
# Estadístico KS (Kolmogorov-Smirnov) — muy usado en scoring bancario
from scipy import stats

prob_buenos = prob_test[y_test == 0]
prob_malos  = prob_test[y_test == 1]

ks_stat, ks_pval = stats.ks_2samp(prob_buenos, prob_malos)
print(f'=== ESTADÍSTICO KS ===')
print(f'KS = {ks_stat:.4f}  |  p-valor = {ks_pval:.4e}')
print()
if ks_stat > 0.4:
    print('✅ KS > 0.40: Modelo con buena capacidad discriminante.')
elif ks_stat > 0.2:
    print('⚠️  KS entre 0.20 y 0.40: Discriminación aceptable pero mejorable.')
else:
    print('❌ KS < 0.20: Modelo con baja capacidad discriminante.')

# Distribución de scores por grupo
plt.figure(figsize=(9, 4))
plt.hist(prob_buenos, bins=25, alpha=0.6, color='#1a6bab', label='Buenos pagadores', density=True)
plt.hist(prob_malos,  bins=25, alpha=0.6, color='#d64045', label='Malos pagadores',  density=True)
plt.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Umbral 0.5')
plt.xlabel('Probabilidad de Default (Score)')
plt.ylabel('Densidad')
plt.title('Distribución de Scores por Grupo', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Regresión OLS y Diagnóstico Econométrico

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor

# OLS: modelar monto del crédito en función de otras variables
# (útil para entender qué factores determinan el crédito otorgado)
X_ols = df_model[['duration', 'installment_commitment', 'age',
                   'existing_credits', 'num_dependents']]
y_ols = df_model['credit_amount']

X_ols_c = sm.add_constant(X_ols)  # agregar intercepto
modelo_ols = sm.OLS(y_ols, X_ols_c).fit()
print(modelo_ols.summary())

In [ ]:
print('=== DIAGNÓSTICO DE SUPUESTOS MCO ===')
print()

# 1. Heterocedasticidad — Breusch-Pagan
bp_lm, bp_pval, bp_f, bp_fpval = het_breuschpagan(modelo_ols.resid, modelo_ols.model.exog)
print(f'1. Heterocedasticidad (Breusch-Pagan):')
print(f'   LM = {bp_lm:.4f}  |  p-valor = {bp_pval:.4f}')
print(f'   {"❌ Heterocedasticidad presente" if bp_pval < 0.05 else "✅ Homocedasticidad no rechazada"}')
print()

# 2. Autocorrelación — Durbin-Watson
dw = durbin_watson(modelo_ols.resid)
print(f'2. Autocorrelación (Durbin-Watson):')
print(f'   DW = {dw:.4f}  (esperado ≈ 2.0 si no hay autocorrelación)')
if dw < 1.5:
    print('   ❌ Autocorrelación positiva detectada')
elif dw > 2.5:
    print('   ❌ Autocorrelación negativa detectada')
else:
    print('   ✅ Sin evidencia de autocorrelación')
print()

# 3. Multicolinealidad — VIF
print('3. Multicolinealidad (VIF):')
vif_data = pd.DataFrame()
vif_data['Variable'] = X_ols.columns
vif_data['VIF'] = [variance_inflation_factor(X_ols.values, i)
                   for i in range(X_ols.shape[1])]
vif_data['Diagnóstico'] = vif_data['VIF'].apply(
    lambda v: '✅ OK' if v < 5 else ('⚠️  Moderado' if v < 10 else '❌ Severo')
)
print(vif_data.to_string(index=False))
print()

# 4. Normalidad de residuos — Jarque-Bera
jb_stat, jb_pval, skew, kurt = sm.stats.stattools.jarque_bera(modelo_ols.resid)
print(f'4. Normalidad residuos (Jarque-Bera):')
print(f'   JB = {jb_stat:.4f}  |  p-valor = {jb_pval:.4f}')
print(f'   Asimetría = {skew:.4f}  |  Curtosis = {kurt:.4f}')
print(f'   {"❌ Residuos no normales" if jb_pval < 0.05 else "✅ Normalidad no rechazada"}')

In [ ]:
# Gráficos de diagnóstico de residuos
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

fitted = modelo_ols.fittedvalues
resid  = modelo_ols.resid

# 1. Residuos vs Ajustados
axes[0,0].scatter(fitted, resid, alpha=0.4, color='#1a6bab', s=20)
axes[0,0].axhline(0, color='red', linestyle='--', lw=1.5)
axes[0,0].set_xlabel('Valores ajustados')
axes[0,0].set_ylabel('Residuos')
axes[0,0].set_title('Residuos vs Ajustados', fontweight='bold')

# 2. Q-Q Plot
from scipy.stats import probplot
(osm, osr), (slope, intercept, r) = probplot(resid, dist='norm')
axes[0,1].scatter(osm, osr, alpha=0.4, color='#1a6bab', s=20)
axes[0,1].plot(osm, slope*np.array(osm)+intercept, 'r--', lw=1.5)
axes[0,1].set_xlabel('Cuantiles teóricos')
axes[0,1].set_ylabel('Cuantiles observados')
axes[0,1].set_title('Q-Q Plot de Residuos', fontweight='bold')

# 3. Histograma de residuos
axes[1,0].hist(resid, bins=30, color='#1a6bab', alpha=0.7, edgecolor='white')
axes[1,0].set_xlabel('Residuos')
axes[1,0].set_ylabel('Frecuencia')
axes[1,0].set_title('Distribución de Residuos', fontweight='bold')

# 4. Scale-Location
axes[1,1].scatter(fitted, np.sqrt(np.abs(resid)), alpha=0.4, color='#1a6bab', s=20)
axes[1,1].set_xlabel('Valores ajustados')
axes[1,1].set_ylabel('√|Residuos|')
axes[1,1].set_title('Scale-Location (Homocedasticidad)', fontweight='bold')

plt.tight_layout()
plt.suptitle('Diagnóstico de Supuestos MCO', fontsize=14, fontweight='bold', y=1.02)
plt.show()

## 6. Series de Tiempo — Índice de Mora Simulado

In [ ]:
# Simular una serie de tiempo mensual de índice de mora (realista para un banco)
# Esta es la estructura típica que encontrarás en el banco
np.random.seed(42)
n = 60  # 5 años de datos mensuales
fechas = pd.date_range(start='2019-01-01', periods=n, freq='ME')

# Índice de mora con tendencia + estacionalidad + ruido
tendencia    = np.linspace(0.03, 0.06, n)                         # sube de 3% a 6%
estacional   = 0.005 * np.sin(2 * np.pi * np.arange(n) / 12)    # ciclo anual
shock_covid  = np.where((fechas >= '2020-03-01') & (fechas <= '2021-06-01'), 0.015, 0)
ruido        = np.random.normal(0, 0.002, n)

mora = tendencia + estacional + shock_covid + ruido
mora = np.clip(mora, 0.01, 0.15)

serie_mora = pd.Series(mora, index=fechas, name='indice_mora')

# Visualizar
fig, axes = plt.subplots(2, 1, figsize=(13, 7))

axes[0].plot(serie_mora.index, serie_mora.values * 100, color='#1a6bab', lw=2)
axes[0].fill_between(serie_mora.index, serie_mora.values * 100, alpha=0.15, color='#1a6bab')
axes[0].axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-06-01'),
                alpha=0.1, color='red', label='Shock COVID-19')
axes[0].set_title('Índice de Mora Mensual (%)', fontweight='bold')
axes[0].set_ylabel('% Mora')
axes[0].legend()

# Descomposición
from statsmodels.tsa.seasonal import seasonal_decompose
decomp = seasonal_decompose(serie_mora, model='additive', period=12)
decomp.trend.plot(ax=axes[1], color='#d64045', lw=2, label='Tendencia')
axes[1].set_title('Componente de Tendencia', fontweight='bold')
axes[1].set_ylabel('% Mora')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Test de estacionariedad ADF
resultado_adf = adfuller(serie_mora)
print('=== TEST ADF (Augmented Dickey-Fuller) ===')
print(f'Estadístico ADF : {resultado_adf[0]:.4f}')
print(f'p-valor         : {resultado_adf[1]:.4f}')
print(f'Valores críticos: {resultado_adf[4]}')
print()
if resultado_adf[1] < 0.05:
    print('✅ p < 0.05 → Se rechaza H0 → La serie ES estacionaria')
    print('   Podemos aplicar ARIMA con d=0')
else:
    print('❌ p > 0.05 → No se rechaza H0 → La serie NO es estacionaria')
    print('   Hay que diferenciar (d=1) y repetir el test')

# Gráficos ACF y PACF
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(serie_mora, lags=24, ax=axes[0], color='#1a6bab')
axes[0].set_title('ACF — Autocorrelación', fontweight='bold')
plot_pacf(serie_mora, lags=24, ax=axes[1], color='#1a6bab', method='ywm')
axes[1].set_title('PACF — Autocorrelación Parcial', fontweight='bold')
plt.tight_layout()
plt.show()

print('\n💡 Cómo leer ACF/PACF para elegir p y q en ARIMA:')
print('   - PACF corta en el rezago p → orden AR = p')
print('   - ACF corta en el rezago q  → orden MA = q')

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Ajustar ARIMA
# Usamos la serie diferenciada si no era estacionaria
modelo_arima = ARIMA(serie_mora, order=(1, 1, 1))
fit_arima = modelo_arima.fit()
print(fit_arima.summary())

# Proyección 12 meses
forecast = fit_arima.get_forecast(steps=12)
pred_mean = forecast.predicted_mean
pred_ci   = forecast.conf_int(alpha=0.05)

# Graficar
plt.figure(figsize=(13, 5))
plt.plot(serie_mora.index, serie_mora.values * 100, color='#1a6bab', lw=2, label='Observado')
plt.plot(pred_mean.index, pred_mean.values * 100, color='#d64045', lw=2,
         linestyle='--', label='Proyección ARIMA(1,1,1)')
plt.fill_between(pred_ci.index,
                 pred_ci.iloc[:, 0] * 100,
                 pred_ci.iloc[:, 1] * 100,
                 alpha=0.2, color='#d64045', label='IC 95%')
plt.axvline(serie_mora.index[-1], color='gray', linestyle=':', lw=1.5)
plt.title('Proyección del Índice de Mora — ARIMA(1,1,1)', fontweight='bold')
plt.ylabel('% Mora')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Pérdida Esperada (EL = PD × LGD × EAD)

In [ ]:
# Calcular la Pérdida Esperada para la cartera de prueba
# PD  = probabilidad predicha por el modelo logístico
# LGD = supuesto del sector (40% sin garantía, 20% con garantía)
# EAD = monto del crédito

cartera = X_test.copy()
cartera['PD']     = prob_test                                      # del modelo logístico
cartera['EAD']    = df_model.loc[X_test.index, 'credit_amount']   # monto real
cartera['LGD']    = 0.40                                           # supuesto regulatorio
cartera['EL']     = cartera['PD'] * cartera['LGD'] * cartera['EAD']
cartera['real']   = y_test.values

print('=== CARTERA DE PRUEBA — PÉRDIDA ESPERADA ===')
print(f'Número de créditos         : {len(cartera):,}')
print(f'EAD total                  : ${cartera["EAD"].sum():,.0f} DM')
print(f'PD promedio                : {cartera["PD"].mean():.2%}')
print(f'Pérdida Esperada total     : ${cartera["EL"].sum():,.0f} DM')
print(f'EL / EAD (tasa provisión)  : {cartera["EL"].sum() / cartera["EAD"].sum():.2%}')
print()

# Segmentar por nivel de riesgo
cartera['segmento'] = pd.cut(cartera['PD'],
                              bins=[0, 0.2, 0.4, 0.6, 1.0],
                              labels=['Bajo riesgo', 'Riesgo medio', 'Riesgo alto', 'Crítico'])

resumen = cartera.groupby('segmento', observed=True).agg(
    clientes=('PD', 'count'),
    PD_prom=('PD', 'mean'),
    EAD_total=('EAD', 'sum'),
    EL_total=('EL', 'sum')
).round(2)
resumen['EL/EAD'] = (resumen['EL_total'] / resumen['EAD_total']).map('{:.2%}'.format)
resumen['PD_prom'] = resumen['PD_prom'].map('{:.2%}'.format)
print(resumen.to_string())

In [ ]:
# Visualización final: dashboard de riesgo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Distribución de PD
axes[0].hist(cartera['PD'], bins=20, color='#1a6bab', alpha=0.8, edgecolor='white')
axes[0].axvline(cartera['PD'].mean(), color='red', lw=2, linestyle='--',
                label=f'Media: {cartera["PD"].mean():.2%}')
axes[0].set_title('Distribución de PD', fontweight='bold')
axes[0].set_xlabel('Probabilidad de Default')
axes[0].legend()

# 2. EAD por segmento
seg = cartera.groupby('segmento', observed=True)['EAD'].sum()
colors_seg = ['#1a6bab', '#5ba3d9', '#f5a623', '#d64045']
axes[1].bar(seg.index, seg.values, color=colors_seg)
axes[1].set_title('EAD por Segmento de Riesgo', fontweight='bold')
axes[1].set_ylabel('Monto total (DM)')
axes[1].tick_params(axis='x', rotation=15)

# 3. EL por segmento
el_seg = cartera.groupby('segmento', observed=True)['EL'].sum()
axes[2].bar(el_seg.index, el_seg.values, color=colors_seg)
axes[2].set_title('Pérdida Esperada por Segmento', fontweight='bold')
axes[2].set_ylabel('EL total (DM)')
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle('Dashboard de Riesgo de Crédito — German Credit Dataset',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n🎯 RESUMEN EJECUTIVO:')
print(f'  - El modelo logístico alcanza AUC = {auc:.3f}')
print(f'  - La PD promedio de la cartera es {cartera["PD"].mean():.1%}')
print(f'  - La tasa de provisión recomendada es {cartera["EL"].sum() / cartera["EAD"].sum():.2%}')
print(f'  - Los segmentos "Riesgo alto" y "Crítico" concentran el mayor riesgo relativo')

---
## 📌 Cheat sheet — conceptos clave para la entrevista

| Concepto | Qué es | Cómo calcularlo |
|---|---|---|
| **PD** | Probabilidad de incumplimiento | Regresión logística → predict_proba |
| **LGD** | % pérdida si hay default | Supuesto regulatorio o modelo separado |
| **EAD** | Monto expuesto al momento del default | Saldo + intereses devengados |
| **EL** | Pérdida esperada | PD × LGD × EAD |
| **AUC-ROC** | Discriminación del modelo | sklearn roc_auc_score |
| **KS** | Separación entre buenos y malos | scipy.stats.ks_2samp |
| **VIF** | Multicolinealidad | variance_inflation_factor |
| **DW** | Autocorrelación de residuos | durbin_watson (≈2 = OK) |
| **ADF** | Estacionariedad de serie | adfuller (p<0.05 = estacionaria) |
| **ARIMA(p,d,q)** | Modelo de serie temporal | ACF/PACF → elegir p y q |
